# Jupyter Notebook reading and matching the KVK data with the CBS SBI classifications
## Filtering SBI codes for third places

## ------------------------------------
## Block 0 - packages
## ------------------------------------

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

## ------------------------------------
## Block 1 - Where the data lives (set ROOT once)
## ------------------------------------

In [ ]:
# Read the 2024 KVK data from Firm backbone
ROOT = Path("S:/")
file_path = ROOT / "the_tentative_team/raw_data/FIRMBACKBONE/kvk_202402_complete/part-00000-4e033ad5-190a-479d-a0ca-75405760d44d-c000.snappy.parquet"
export_path = ROOT / "the_tentative_team/results/"
firmdata_raw = pd.read_parquet(file_path)

In [ ]:
# read the codebook for sbi codes, see codebook.csv
# 0 : no third place
# 1 : food and beverage
# 2 : retail 
# 3 : community services
# 4 : cultural venues
# 5 : sport and leisure
# 6 : medical services
# 7 : shopping malls
# 8 : department stores

codebook_path = ROOT / "the_tentative_team/processed_data/sbi_codebook.csv"
codes = pd.read_csv(codebook_path)
codebook = codes[["code", "name", "classification"]]


## ------------------------------------
## Block 2 - Data Validation and Data Processing
## ------------------------------------

In [ ]:
# some codes are non-unique
len(codebook["code"].unique())

In [ ]:
# remove duplicated codes 
a = codebook[["code", "classification"]].drop_duplicates(subset = "code", keep = "first")
a = pd.DataFrame(a)

In [ ]:
# looks good now
len(a)

In [ ]:
# define codebook as a map / dictionary
codebook = a.drop_duplicates(subset = "code", keep = "first")
code_to_class = dict(zip(codebook["code"], codebook["classification"]))
len(a)

In [ ]:
# create a subset of the firmdata only containing the relevant data
firmdata = firmdata_raw[["fbb_id", "code_sbi_1", "code_sbi_2", "code_sbi_3", "establishment_pc4"]].copy()

# for the primary, secondary, and tertiary sbi classification, map the corresponding third place classification from codebook.csv
for col in ["code_sbi_1", "code_sbi_2", "code_sbi_3"]:
    firmdata[col + "_class"] = firmdata[col].map(code_to_class)

In [ ]:
# now, create one single classification value: the first non-zero, non-missing value of sbi_1, _2, and _3
# this is because companies have primary (sbi_1), secondary (sbi_2), and tertiary (sbi_3) classifications
# we take the highest non-zero, non-missing classification for the company

firmdata["classification"] = (firmdata[["code_sbi_1_class", "code_sbi_2_class", "code_sbi_3_class"]]
                             .replace(0, np.nan)
                              .bfill(axis = 1)
                              .iloc[:, 0]
                             )

In [ ]:
print(firmdata["classification"].value_counts())
firmdata = firmdata.dropna(subset = ["classification"])

In [ ]:
print(f"We went from a total of {len(firmdata_raw)} to {len(firmdata)} companies.")

In [ ]:
# reduce the data to the relevant columns
data = firmdata[["fbb_id", "establishment_pc4", "classification"]].copy()

## ------------------------------------
## Block 3 - Calulation of amount/number of Third Places per PC4
## ------------------------------------

In [ ]:
# group data by pc4 and calculate the scores
pc4_data = (pd.crosstab(data["establishment_pc4"], data["classification"])
            .reindex(columns = range(1, 9), fill_value = 0)
            .rename(columns = {i: f"classification_{i}_count" for i in range(1,9)})
            .reset_index()
           )

pc4_data = pc4_data.iloc[1:, 0:]

In [ ]:
pc4_data.head()

In [ ]:
# Calculate the sum of third places in a pc4 by category

# total sum
class_cols = [f"classification_{i}_count" for i in range(1,9)]
pc4_data["third_places_total"] = pc4_data[class_cols].sum(axis = 1)

# total sum excluding medical services
class_cols = [f"classification_{i}_count" for i in (1,2,3,4,5,7,8)]
pc4_data["third_places_non_medical_total"] = pc4_data[class_cols].sum(axis = 1)

# total sum excluding medical services
class_cols = [f"classification_{i}_count" for i in (1,3,4,5)]
pc4_data["third_places_non_medical_non_retail_total"] = pc4_data[class_cols].sum(axis = 1)

# sum of medical services
pc4_data["third_places_medical"] = pc4_data["classification_6_count"]

# sum of food and beverage places
pc4_data["third_places_food_beverage"] = pc4_data["classification_1_count"]

# retail has three categories (1 : normal, 7: malls, 8: department stores)
class_cols = [f"classification_{i}_count" for i in (2,7,8)]
pc4_data["third_places_retail"] = pc4_data[class_cols].sum(axis = 1)

# non-commercial community splaces
pc4_data["third_places_community"] = pc4_data["classification_3_count"]

# cultural venues
pc4_data["third_places_cultural"] = pc4_data["classification_4_count"]

# sport and leisure
pc4_data["third_places_sport_leisure"] = pc4_data["classification_5_count"]



In [ ]:
# this seems to have worked
print(pc4_data.head())
print(pc4_data["third_places_total"].value_counts())

In [ ]:
# double check the classification counts, yes -- seems to work
data["classification"].value_counts()

In [ ]:
# some postcodes seem to be longer than 4
(pc4_data["establishment_pc4"].astype(str).str.len() > 4).sum()

# create an indicator so we can change them
pc4_data["longer_than_4"] = (
    pc4_data["establishment_pc4"].astype(str).str.len() > 4
).astype(int)

print(pc4_data["establishment_pc4"][pc4_data["longer_than_4"] == 1])


In [ ]:
# replace 5-code pc4 by 4-code: only take the first 4 ciphers

pc4_data.loc[pc4_data["longer_than_4"] == 1, "establishment_pc4"] = (pc4_data.loc[pc4_data["longer_than_4"] == 1, "establishment_pc4"].astype(str).str[:4].astype(int))
    

In [ ]:
print(pc4_data["establishment_pc4"][pc4_data["longer_than_4"] == 1])


In [ ]:
# subset the data to the relevant columns

pc4_data = pc4_data[["establishment_pc4", "third_places_total", "third_places_non_medical_total", "third_places_medical", "third_places_food_beverage", "third_places_retail", "third_places_community", "third_places_cultural", "third_places_sport_leisure", "third_places_non_medical_non_retail_total"]]

In [ ]:
pc4_data.head()

In [ ]:
# export locally (s drive)
pc4_data.to_csv(export_path / "pc4_data.csv", index = False)

In [ ]:
len(pc4_data)

In [ ]:
len(pc4_data["establishment_pc4"].unique())

## ------------------------------------
## (Optional) Block 4 - Calulation of amount/number of Third Places per PC3
## ------------------------------------

In [ ]:
pc3_data = pc4_data.copy()

In [ ]:
pc3_data["pc3"] = pc3_data["establishment_pc4"].str[:3]

In [ ]:
print(pc3_data["pc3"].head())
print(pc3_data["establishment_pc4"].head())

In [ ]:
# group data by pc3 and calculate the scores

cols_to_sum = ["third_places_total", "third_places_non_medical_total", "third_places_medical", "third_places_food_beverage", "third_places_retail", "third_places_community", "third_places_cultural", "third_places_sport_leisure", "third_places_non_medical_non_retail_total"]

pc3_aggregated = (
    pc3_data.groupby("pc3", as_index = False)[cols_to_sum]
    .sum()
)

In [ ]:
# group data by pc3 and calculate the scores

print(len(pc3_aggregated))
print(pc3_aggregated.head())


In [ ]:
# export locally (c drive)
pc3_aggregated.to_csv("C:/Users/thovestadt/Documents/the_tentative_team/data_export/pc3_data_number_third_places.csv", index = False)

In [ ]:
# export locally (s drive)
pc3_aggregated.to_csv(export_path / "pc3_data_number_third_places.csv", index = False)

## ------------------------------------
## (Optional) Block 5 - Calulation of amount/number of Third Places per PC2
## ------------------------------------

In [ ]:
pc2_data = pc3_aggregated.copy()

In [ ]:
pc2_data["pc2"] = pc2_data["pc3"].str[:2]

In [ ]:
print(pc2_data["pc3"].head())
print(pc2_data["pc2"].head())

In [ ]:
# group data by pc3 and calculate the scores

cols_to_sum = ["third_places_total", "third_places_non_medical_total", "third_places_medical", "third_places_food_beverage", "third_places_retail", "third_places_community", "third_places_cultural", "third_places_sport_leisure", "third_places_non_medical_non_retail_total"]

pc2_aggregated = (
    pc2_data.groupby("pc2", as_index = False)[cols_to_sum]
    .sum()
)

In [ ]:

print(len(pc2_aggregated))
print(pc2_aggregated.head())


In [ ]:
# export locally (c drive)
pc2_aggregated.to_csv(export_path / "pc2_data_number_third_places.csv", index = False)

In [ ]:
# export to cloud (s drive)
pc2_aggregated.to_csv(export_path / "pc2_data_number_third_places.csv", index = False)